In [23]:
#!/usr/bin/env python3
"""
log ファイルから
    • 訓練 loss
    • eval_loss
    • learning_rate
を抽出し，個別に PNG で保存する。

前提:
    - 1 行 1 レコード形式で，行は Python dict 文字列
    - ファイルは UTF‑8 テキスト
"""

'\nlog ファイルから\n    • 訓練 loss\n    • eval_loss\n    • learning_rate\nを抽出し，個別に PNG で保存する。\n\n前提:\n    - 1 行 1 レコード形式で，行は Python dict 文字列\n    - ファイルは UTF‑8 テキスト\n'

In [24]:
import ast
from pathlib import Path
import matplotlib.pyplot as plt
from itertools import cycle
import numpy as np

In [25]:
BASE_PATH = Path.cwd().parent

In [26]:
file = "train_log_20250715_213719_epoch15_5fold.log"
file = "train_log_20250715_130336_epoch3_5fold.log"

suffix = "5fold_3epoch"

In [27]:
# ==== 入力・出力設定 ====
log_path = Path(BASE_PATH / "logs" / file)  # 元ログ
out_dir = Path(BASE_PATH / "plots")  # 画像出力ディレクトリ
out_dir.mkdir(exist_ok=True)

# ── データ格納 (foldごとにリストのリスト) ───────────────────────────
folds_train_ep, folds_train_loss = [], []
folds_eval_ep, folds_eval_loss = [], []
folds_lr_ep, folds_lr = [], []
conf_matrices = []  # NEW

fold_idx = -1
reading_cf = False  # NEW
cf_rows = []  # NEW

In [28]:
# ───── ログ読み取り ──────────────────────────────
with log_path.open("r", encoding="utf-8") as f:
    for raw in f:
        line = raw.rstrip()

        # ----- fold 切り替え -----
        if line.startswith("🔄 Start") and "fold" in line:
            fold_idx += 1
            folds_train_ep.append([])
            folds_train_loss.append([])
            folds_eval_ep.append([])
            folds_eval_loss.append([])
            folds_lr_ep.append([])
            folds_lr.append([])
            continue

        # ----- confusion matrix 読み取り開始 -----
        if "confusion_matrix" in line:
            reading_cf = True
            cf_rows = []
            continue

        if reading_cf:
            # 行トリム & 終端判定
            row_str = line.strip()
            if not row_str:
                continue
            # 行末 ]]
            if row_str.endswith("]]"):
                row_str = row_str[:-2].strip()  # 末尾 ]]
                last_row = True
            else:
                last_row = False

            # [ 3  2  0  0  0] → list[int]
            row_numbers = [int(x) for x in row_str.lstrip("[").rstrip("]").split()]
            cf_rows.append(row_numbers)

            if last_row:
                conf_matrices.append(np.array(cf_rows, dtype=int))
                reading_cf = False
            continue

        # ----- dict 行のみパース -----
        if line.lstrip().startswith("{"):
            try:
                rec = ast.literal_eval(line)
            except (SyntaxError, ValueError):
                continue

            if "loss" in rec and "eval_loss" not in rec:
                folds_train_ep[fold_idx].append(rec["epoch"])
                folds_train_loss[fold_idx].append(rec["loss"])
                folds_lr_ep[fold_idx].append(rec["epoch"])
                folds_lr[fold_idx].append(rec["learning_rate"])
            elif "eval_loss" in rec:
                folds_eval_ep[fold_idx].append(rec["epoch"])
                folds_eval_loss[fold_idx].append(rec["eval_loss"])

In [29]:
# ───── プロット関数 ───────────────────────────────
def plot_folds(x_folds, y_folds, xlabel, ylabel, title, fname):
    plt.figure()
    color_cycle = cycle(plt.rcParams["axes.prop_cycle"].by_key()["color"])
    for i, (x, y) in enumerate(zip(x_folds, y_folds)):
        if not x:
            continue
        plt.plot(x, y, label=f"fold {i}", color=next(color_cycle))
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title)
    plt.legend()
    plt.grid(alpha=0.4)
    plt.tight_layout()
    plt.savefig(out_dir / fname, dpi=150)
    plt.close()

In [30]:
# ── グラフ作成 ────────────────────────────────────────────────────
plot_folds(
    folds_train_ep,
    folds_train_loss,
    "epoch",
    "train loss",
    "Training Loss",
    f"train_loss{suffix}.svg",
)

plot_folds(
    folds_eval_ep,
    folds_eval_loss,
    "epoch",
    "eval loss",
    "Evaluation Loss",
    f"eval_loss{suffix}.svg",
)

plot_folds(
    folds_lr_ep,
    folds_lr,
    "epoch",
    "learning rate",
    "Learning Rate Schedule per Fold",
    f"learning_rate{suffix}.svg",
)

print(f"✅ 生成完了: {out_dir.resolve()} に *{suffix}.png が保存されました。")

✅ 生成完了: /home/masa1357/Dockerdata/gitfile/LLaVA-FT-datikz_Taiga/plots に *5fold_3epoch.png が保存されました。


In [31]:
conf_matrices

[array([[ 41,   0,   0,   0,   0],
        [107,  72,   0,   0,   0],
        [ 28,   0,  74,   0,   0],
        [  0,   0,   0,  19,   0],
        [  2,   0,   0,   0,  34]]),
 array([[ 7,  2,  0,  0,  0],
        [22,  9,  8,  0,  0],
        [ 4,  7, 10,  0,  0],
        [ 1,  0,  3,  0,  0],
        [ 1,  0,  0,  0,  2]]),
 array([[ 4,  1,  2,  0,  0],
        [21, 11,  3,  0,  1],
        [11,  4,  4,  0,  3],
        [ 0,  2,  1,  0,  0],
        [ 0,  1,  0,  0,  7]]),
 array([[ 7,  1,  2,  0,  0],
        [22,  5,  6,  0,  2],
        [ 3,  7,  8,  0,  1],
        [ 0,  0,  2,  0,  0],
        [ 1,  0,  1,  0,  7]]),
 array([[ 9,  1,  0,  0,  0],
        [15, 14,  4,  0,  0],
        [ 6,  8,  7,  0,  0],
        [ 0,  1,  4,  0,  2],
        [ 0,  0,  0,  0,  4]]),
 array([[ 3,  2,  0,  0,  0],
        [25,  4,  6,  0,  0],
        [ 5,  7,  6,  0,  2],
        [ 0,  1,  2,  0,  1],
        [ 1,  0,  1,  0,  9]])]

In [ ]:
# conf_matrices = conf_matrices[1:]
conf_matrices

[array([[ 7,  2,  0,  0,  0],
        [22,  9,  8,  0,  0],
        [ 4,  7, 10,  0,  0],
        [ 1,  0,  3,  0,  0],
        [ 1,  0,  0,  0,  2]]),
 array([[ 4,  1,  2,  0,  0],
        [21, 11,  3,  0,  1],
        [11,  4,  4,  0,  3],
        [ 0,  2,  1,  0,  0],
        [ 0,  1,  0,  0,  7]]),
 array([[ 7,  1,  2,  0,  0],
        [22,  5,  6,  0,  2],
        [ 3,  7,  8,  0,  1],
        [ 0,  0,  2,  0,  0],
        [ 1,  0,  1,  0,  7]]),
 array([[ 9,  1,  0,  0,  0],
        [15, 14,  4,  0,  0],
        [ 6,  8,  7,  0,  0],
        [ 0,  1,  4,  0,  2],
        [ 0,  0,  0,  0,  4]]),
 array([[ 3,  2,  0,  0,  0],
        [25,  4,  6,  0,  0],
        [ 5,  7,  6,  0,  2],
        [ 0,  1,  2,  0,  1],
        [ 1,  0,  1,  0,  9]])]

In [33]:
# --- 合算 CM -----------------------------------------------------------
cm_total = np.sum(conf_matrices, axis=0)

# ===== 行正規化して確率 [%] に変換 =====
row_sum = cm_total.sum(axis=1, keepdims=True)
cm_percent = np.divide(cm_total, row_sum, where=row_sum != 0) * 100  # shape [C,C]

# --------- 指標計算 ----------------------------------------------------
acc = np.trace(cm_total) / cm_total.sum()

tp = np.diag(cm_total)
fp = cm_total.sum(axis=0) - tp
fn = cm_total.sum(axis=1) - tp
precision = np.where(tp + fp == 0, 0, tp / (tp + fp))
recall = np.where(tp + fn == 0, 0, tp / (tp + fn))
f1_each = np.where(
    precision + recall == 0, 0, 2 * precision * recall / (precision + recall)
)
macro_f1 = f1_each.mean()

print("===== Confusion Matrix (sum of folds) =====")
print(cm_total)
print(f"Overall Accuracy : {acc:.4f}")
print(f"Macro‑average F1 : {macro_f1:.4f}")

# --------- 可視化 ------------------------------------------------------
class_names = ["A", "B", "C", "D", "F"]  # ← ここで対応表を用意

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm_percent, cmap="Blues")

# 軸ラベルを文字列に置換
ax.set_xticks(np.arange(len(class_names)), labels=class_names)
ax.set_yticks(np.arange(len(class_names)), labels=class_names)

ax.set_title("Confusion Matrix")
ax.set_xlabel("Predicted label")
ax.set_ylabel("True label")
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Percentage (%)")

# セルごとの % 表示
for (i, j), val in np.ndenumerate(cm_percent):
    ax.text(
        j,
        i,
        f"{val:.2f}%",
        ha="center",
        va="center",
        color="white" if val > cm_percent.max() * 0.6 else "black",
        fontsize=8,
    )

plt.tight_layout()
plt.savefig(out_dir / f"confusion_matrix{suffix}.svg", dpi=150)
plt.close()

===== Confusion Matrix (sum of folds) =====
[[ 30   7   4   0   0]
 [105  43  27   0   3]
 [ 29  33  35   0   6]
 [  1   4  12   0   3]
 [  3   1   2   0  29]]
Overall Accuracy : 0.3634
Macro‑average F1 : 0.3512


/tmp/ipykernel_3516876/2467966212.py:14: RuntimeWarning: invalid value encountered in divide
  precision = np.where(tp + fp == 0, 0, tp / (tp + fp))
/tmp/ipykernel_3516876/2467966212.py:17: RuntimeWarning: invalid value encountered in divide
  precision + recall == 0, 0, 2 * precision * recall / (precision + recall)
